In [0]:
%sql
SELECT *
FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/delivery_performance/`
ORDER BY shipment_id
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_shipments,

    SUM(
        CASE
            WHEN is_delivered = true THEN 1
            ELSE 0
        END
    ) AS delivered_shipments,

    SUM(
        CASE
            WHEN is_late = true THEN 1
            ELSE 0
        END
    ) AS late_shipments,

    ROUND(
        AVG(delivery_duration_hours),
        2
    ) AS avg_delivery_duration_hours,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_delivered = true
                 AND is_late = false
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN is_delivered = true
                    THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS on_time_delivery_rate_pct

FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/delivery_performance/`;

In [0]:
%sql
SELECT
    carrier_id,
    carrier_name,
    total_shipments,
    delivered_shipments,
    late_shipments,
    on_time_shipments,

    ROUND(
        avg_delivery_duration_hours,
        2
    ) AS avg_delivery_duration_hours,

    on_time_delivery_rate_pct

FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/carrier_performance/`

ORDER BY on_time_delivery_rate_pct DESC;

In [0]:
%sql
SELECT
    shipment_date,
    total_shipments,
    delivered_shipments,
    late_shipments,
    on_time_shipments,

    ROUND(
        avg_delivery_duration_hours,
        2
    ) AS avg_delivery_duration_hours,

    on_time_delivery_rate_pct

FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/daily_shipments/`

ORDER BY shipment_date;

In [0]:
%sql
SELECT
    shipment_id,
    order_id,
    carrier_name,
    origin_city,
    destination_city,
    expected_delivery_time,
    delivered_at,
    delivery_delay_hours

FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/late_shipments/`

ORDER BY delivery_delay_hours DESC

LIMIT 10;

In [0]:
%sql
SELECT
    carrier_name,

    COUNT(*) AS late_shipments,

    ROUND(
        AVG(delivery_delay_hours),
        2
    ) AS avg_delay_hours,

    ROUND(
        MAX(delivery_delay_hours),
        2
    ) AS max_delay_hours

FROM delta.`abfss://logistics@stsupplychain2026xx.dfs.core.windows.net/gold/late_shipments/`

GROUP BY carrier_name

ORDER BY late_shipments DESC;